[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/02-o-que-e-ml/notebooks/02_demo.ipynb)

# Aula 2 — Demonstração: do problema ao primeiro modelo
*Computação Aplicada à Economia: Tópicos em Aprendizado de Máquina*

Vamos percorrer o fluxo completo de um projeto com uma base clássica de preços de imóveis:

**pergunta → dados → separar → treinar → avaliar → comunicar**

Pelo caminho, vários gráficos: cada um mostra um recurso do Python ou do matplotlib para deixar a figura clara e bonita. Na **Parte 2**, voltamos ao gato e ao cachorro dos slides.

> A base é americana (censo da Califórnia, 1990). Usamos porque é limpa e pública; o objetivo hoje é aprender o *fluxo*.

## 1. A pergunta (ficha de formulação)

| Item | Resposta |
|---|---|
| **Decisão** | precificar imóveis em regiões sem transações recentes |
| **Unidade** | um quarteirão censitário |
| **Alvo y** | valor mediano dos imóveis (em US$ 100 mil) |
| **Atributos x** | renda mediana, idade dos imóveis, cômodos, população, localização |
| **Tarefa** | regressão |
| **Métrica** | erro típico da previsão (RMSE), na mesma unidade de y |

## 0. Configuração visual (uma vez por notebook)
Em vez de repetir cores e ajustes em cada gráfico, definimos tudo **uma vez**. Recursos do Python: **dicionários** (`CORES`, `rcParams`) e o método `.update`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Paleta do curso: um dicionário dá NOME às cores (CORES["laranja"] é mais legível que "#EB811B")
CORES = {"azul": "#2A6F97", "laranja": "#EB811B", "verde": "#14B03D",
         "vermelho": "#B22234", "escuro": "#23373B", "cinza": "#8C8C8C"}

# rcParams guarda o padrão de TODOS os gráficos; também é um dicionário
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.spines.top": False,        # tira as bordas de cima e da direita
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",    # título à esquerda, como em relatórios
    "axes.prop_cycle": plt.cycler(color=list(CORES.values())),
    "legend.frameon": False,
})

## 2. Os dados

In [ ]:
try:
    from sklearn.datasets import fetch_california_housing
    df = fetch_california_housing(as_frame=True).frame
    print("Base carregada:", df.shape)
except Exception as erro:  # sem internet: base sintética com a mesma estrutura
    print("Usando base simulada:", erro)
    rng = np.random.default_rng(0)
    n = 5000
    df = pd.DataFrame({
        "MedInc": rng.lognormal(1.2, 0.45, n), "HouseAge": rng.integers(1, 52, n),
        "AveRooms": rng.normal(5.4, 1.2, n), "AveBedrms": rng.normal(1.1, 0.2, n),
        "Population": rng.lognormal(7, 0.7, n), "AveOccup": rng.normal(3, 0.8, n),
        "Latitude": rng.uniform(32.5, 42, n), "Longitude": rng.uniform(-124.3, -114.3, n)})
    df["MedHouseVal"] = (0.45 * df.MedInc + 0.01 * df.HouseAge
                         - 0.1 * (df.Latitude - 34) + rng.normal(0, 0.6, n)).clip(0.15, 5)

df.head()

In [ ]:
# Cada linha é uma observação (quarteirão); cada coluna, uma variável
alvo = "MedHouseVal"
y = df[alvo]
X = df.drop(columns=alvo)
print(f"n = {X.shape[0]} observações, p = {X.shape[1]} atributos")
X.describe().T[["mean", "std", "min", "max"]]

### Olhar antes de modelar
Recursos: `plt.subplots` com uma **grade** de eixos, `zip` para percorrer eixos e colunas juntos e **f-strings** com formato (`{mediana:,.2f}`).

In [ ]:
# Um painel por variável: zip percorre os eixos e as colunas ao mesmo tempo
fig, axs = plt.subplots(3, 3, figsize=(11, 8))
for ax, col in zip(axs.flat, df.columns):
    dados = df[col].clip(upper=df[col].quantile(0.99))   # corta o 1% extremo só no desenho
    ax.hist(dados, bins=40, color=CORES["azul"], alpha=0.8)
    mediana = df[col].median()
    ax.axvline(mediana, color=CORES["laranja"], lw=2)
    ax.set_title(f"{col}  (mediana = {mediana:,.2f})", fontsize=10)   # f-string com formato
fig.suptitle("Distribuição de cada variável (linha laranja = mediana)", fontweight="bold")
fig.tight_layout()
plt.show()

**Pergunta:** o que há de estranho na cauda direita de `MedHouseVal`?

In [ ]:
teto = df[alvo].max()
print(f"{(df[alvo] >= 5).mean():.1%} dos quarteirões estão no teto da base ({teto:.2f})")

Os valores acima de US$ 500 mil foram registrados como 5,0: a variável é **censurada** no topo.

> **Conexão com a econometria:** é o mesmo problema que motiva o modelo Tobit. Um modelo preditivo também vai sofrer perto do teto: veremos isso no gráfico de previsto × real.

### Renda × valor: quando a dispersão não basta
Com 20 mil pontos, a nuvem vira uma mancha. O **hexbin** conta pontos por hexágono. Recursos: `sharey=True`, `fig.colorbar`, `ax.annotate` com seta.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)   # desempacota os 2 eixos

ax1.scatter(df.MedInc, df[alvo], s=3, alpha=0.2, color=CORES["azul"])
ax1.set(title="Dispersão: pontos demais se sobrepõem",
        xlabel="renda mediana (US$ 10 mil)", ylabel="valor mediano (US$ 100 mil)")

hb = ax2.hexbin(df.MedInc, df[alvo], gridsize=40, cmap="viridis", mincnt=1, bins="log")
fig.colorbar(hb, ax=ax2, label="nº de quarteirões (escala log)")
ax2.set(title="Hexbin: a densidade aparece", xlabel="renda mediana (US$ 10 mil)")
ax2.grid(False)
ax2.annotate("teto da base", xy=(13, 5), xytext=(10.5, 2.8), color=CORES["laranja"],
             fontweight="bold", arrowprops=dict(arrowstyle="->", color=CORES["laranja"]))
fig.tight_layout()
plt.show()

### Um mapa sem mapa
Latitude e longitude já desenham a Califórnia. Recursos: cor e tamanho codificando variáveis (`c=`, `s=`), dicionário de cidades e **desempacotamento de tuplas** no `for`.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
mapa = ax.scatter(df.Longitude, df.Latitude, c=df[alvo], s=df.Population / 400,
                  cmap="magma_r", alpha=0.5)           # cor = preço; tamanho = população
fig.colorbar(mapa, ax=ax, label="valor mediano (US$ 100 mil)", shrink=0.8)

cidades = {"Los Angeles": (-118.24, 34.05), "San Francisco": (-122.42, 37.77)}
for nome, (lon, lat) in cidades.items():               # desempacota a tupla dentro do laço
    ax.annotate(nome, (lon, lat), xytext=(12, 12), textcoords="offset points",
                fontweight="bold", arrowprops=dict(arrowstyle="-"))

ax.set(title="Onde estão os imóveis caros?", xlabel="longitude", ylabel="latitude")
ax.set_aspect(1 / np.cos(np.radians(36)))              # 1° de longitude é mais curto que 1° de latitude
plt.show()

## 3. Separar: o teste fica guardado na gaveta

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
len(X_tr), len(X_te)

## 4–5. Treinar e avaliar (sempre contra uma referência ingênua)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def rmse(modelo, X, y):
    return mean_squared_error(y, modelo.predict(X)) ** 0.5

resultados = {}
for nome, modelo in [("média (ingênuo)", DummyRegressor(strategy="mean")),
                     ("regressão linear", LinearRegression())]:
    modelo.fit(X_tr, y_tr)                       # aprende SÓ com o treino
    resultados[nome] = rmse(modelo, X_te, y_te)  # avalia no teste
pd.Series(resultados, name="RMSE no teste (US$ 100 mil)").round(3)

### Trocar o modelo é trocar uma linha

In [ ]:
from sklearn.ensemble import RandomForestRegressor

floresta = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=0)
floresta.fit(X_tr, y_tr)
resultados["floresta aleatória"] = rmse(floresta, X_te, y_te)
pd.Series(resultados, name="RMSE no teste").round(3)

### Comunicar o resultado: um gráfico vale mais que a tabela
Recursos: **list comprehension** para escolher cores e `ax.bar_label` para escrever os valores.

In [ ]:
res = pd.Series(resultados).sort_values()
# list comprehension: destaca em laranja só o melhor modelo
cores = [CORES["laranja"] if nome == res.idxmin() else CORES["cinza"] for nome in res.index]

fig, ax = plt.subplots(figsize=(7, 2.8))
barras = ax.barh(res.index, res.values, color=cores)
ax.bar_label(barras, fmt="%.3f", padding=4)            # escreve o valor na ponta da barra
ax.set(title="Erro no teste (RMSE, US$ 100 mil): menor é melhor", xlim=(0, res.max() * 1.2))
ax.grid(axis="y", visible=False)
plt.show()

### Previsto × real
A linha tracejada é a previsão perfeita. Repare no **teto em 5**: nenhum modelo sabe o valor verdadeiro desses quarteirões.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, (nome, m) in zip(axs, [("regressão linear", modelo), ("floresta aleatória", floresta)]):
    ax.scatter(y_te, m.predict(X_te), s=4, alpha=0.25)
    ax.plot([0, 5], [0, 5], color=CORES["escuro"], ls="--", lw=1)   # 45°: previsão perfeita
    ax.text(0.04, 0.93, f"RMSE = {rmse(m, X_te, y_te):.3f}", transform=ax.transAxes,
            fontweight="bold", va="top")           # transAxes: posição em fração do painel
    ax.set(title=nome, xlabel="valor real", ylabel="valor previsto")
fig.suptitle("Previsto × real no conjunto de teste", fontweight="bold")
fig.tight_layout()
plt.show()

### Teaser da Aula 3: aprendeu ou decorou?
Compare o erro **no treino** com o erro **no teste**.

In [ ]:
comparacao = pd.DataFrame({
    "treino": [rmse(m, X_tr, y_tr) for m in (modelo, floresta)],
    "teste":  [rmse(m, X_te, y_te) for m in (modelo, floresta)],
}, index=["regressão linear", "floresta aleatória"])

# O pandas desenha direto do DataFrame: uma cor por coluna
ax = comparacao.plot.bar(rot=0, color=[CORES["azul"], CORES["laranja"]], figsize=(6.5, 3.5))
for barras in ax.containers:
    ax.bar_label(barras, fmt="%.2f", padding=2)
ax.set(title="Erro no treino × no teste", ylabel="RMSE")
plt.show()

### Cuidado ao interpretar coeficientes
Os coeficientes abaixo ajudam a **prever**. Eles **não** são efeitos causais: um cômodo a mais não "causa" a variação indicada no preço.

In [ ]:
coefs = pd.Series(modelo.coef_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(coefs.index, coefs.values,
        color=np.where(coefs > 0, CORES["azul"], CORES["vermelho"]))   # cor pelo sinal
ax.axvline(0, color=CORES["escuro"], lw=1)
ax.set(title="Coeficientes da regressão linear (não são efeitos causais!)")
plt.show()

Os coeficientes estão em escalas diferentes (renda em US$ 10 mil, população em pessoas), então nem o **tamanho** das barras é comparável. Prever bem não torna os β interpretáveis.

## Bônus: aprendizado não supervisionado — regiões a partir das coordenadas

In [ ]:
from sklearn.cluster import KMeans

coords = df[["Longitude", "Latitude"]]
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(coords)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(coords.Longitude, coords.Latitude, c=km.labels_, s=2, cmap="tab10")
ax.scatter(*km.cluster_centers_.T, marker="X", s=150, c="black")
ax.set(xlabel="longitude", ylabel="latitude", title="5 grupos propostos pelo k-médias")
plt.show()

Ninguém disse ao algoritmo onde ficam Los Angeles ou São Francisco. Ele propôs grupos de pontos próximos; **nós** interpretamos.

---
# Parte 2 — Mudando de domínio: o gato e o cachorro

Nos slides, $X$ deixou de ser uma tabela de variáveis com significado e virou uma grade de **pixels**. Vamos sentir o problema na prática com imagens **simuladas**: como somos nós que desenhamos, conhecemos o processo gerador por completo.

In [ ]:
N = 24                                   # imagens de 24 x 24 pixels
yy, xx = np.mgrid[0:N, 0:N] + 0.5        # coordenadas do centro de cada pixel


def elipse(cx, cy, rx, ry):
    """Máscara booleana: True dentro da elipse."""
    return ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1


def triangulo(p1, p2, p3):
    """Máscara booleana: True dentro do triângulo p1-p2-p3."""
    def lado(a, b):
        return (xx - b[0]) * (a[1] - b[1]) - (a[0] - b[0]) * (yy - b[1])
    d = np.stack([lado(p1, p2), lado(p2, p3), lado(p3, p1)])
    return ~((d < 0).any(axis=0) & (d > 0).any(axis=0))


def desenhar(classe, dx=0, dy=0, pelo=120):
    """Gato ou cachorro deslocado (dx, dy) pixels. 0 = preto, 255 = branco."""
    im = np.full((N, N), 235.0)                                                       # fundo
    if classe == "gato":
        im[triangulo((4 + dx, 2 + dy), (4 + dx, 12 + dy), (11 + dx, 8 + dy))] = pelo     # orelhas
        im[triangulo((20 + dx, 2 + dy), (20 + dx, 12 + dy), (13 + dx, 8 + dy))] = pelo
        im[elipse(12 + dx, 14 + dy, 8, 6.8)] = pelo                                       # cara
        im[elipse(8.5 + dx, 12.5 + dy, 1.5, 1.5)] = 20                                    # olhos
        im[elipse(15.5 + dx, 12.5 + dy, 1.5, 1.5)] = 20
        im[triangulo((11 + dx, 16 + dy), (13 + dx, 16 + dy), (12 + dx, 17.6 + dy))] = 40  # nariz
    else:
        im[elipse(12 + dx, 12 + dy, 8, 7.6)] = pelo                                       # cara
        im[elipse(4.5 + dx, 13 + dy, 2.6, 5.7)] = 0.55 * pelo                             # orelhas caídas
        im[elipse(19.5 + dx, 13 + dy, 2.6, 5.7)] = 0.55 * pelo
        im[elipse(12 + dx, 17 + dy, 4.8, 3.0)] = min(pelo + 60, 250)                      # focinho
        im[elipse(9 + dx, 10.5 + dy, 1.2, 1.2)] = 20                                      # olhos
        im[elipse(15 + dx, 10.5 + dy, 1.2, 1.2)] = 20
        im[elipse(12 + dx, 15.3 + dy, 1.8, 1.3)] = 25                                     # nariz
    return im


def gerar_imagens(n, desloc=0, aleatorio=False, seed=0):
    """n imagens com ruído; y = 1 para gato. Cada imagem vira UMA LINHA de X (576 colunas).

    desloc: deslocamento horizontal em pixels. Com aleatorio=True, sorteia
    deslocamentos entre -desloc e +desloc, na horizontal e na vertical.
    """
    rng = np.random.default_rng(seed)
    y = rng.integers(0, 2, n)
    X = np.empty((n, N * N))
    for i in range(n):
        if aleatorio:
            dx, dy = rng.integers(-desloc, desloc + 1, size=2)
        else:
            dx, dy = desloc, 0
        im = desenhar("gato" if y[i] == 1 else "cachorro", dx, dy, pelo=rng.uniform(80, 180))
        im = im + rng.normal(0, 25, im.shape)          # ruído de "câmera"
        X[i] = np.clip(im, 0, 255).ravel() / 255        # ravel: grade 24x24 -> linha de 576
    return X, y

In [ ]:
X_img, y_img = gerar_imagens(16, desloc=3, aleatorio=True, seed=3)

fig, axs = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, linha, rotulo in zip(axs.flat, X_img, y_img):
    ax.imshow(linha.reshape(N, N), cmap="gray", vmin=0, vmax=1)   # reshape: linha -> grade
    ax.set_title("gato" if rotulo else "cachorro", fontsize=9,
                 color=CORES["laranja"] if rotulo else CORES["azul"])
    ax.axis("off")
fig.suptitle("16 exemplos: o algoritmo só vê o número de cada pixel", fontweight="bold")
plt.show()

### O logit nos pixels
Cada imagem é uma linha de $X$ com 576 colunas. Vamos ajustar $P(\text{gato}\mid x) = \Lambda(\beta_0 + \sum_j \beta_j x_j)$, exatamente como faríamos com dados de crédito.

In [ ]:
from sklearn.linear_model import LogisticRegression

X_tr_px, y_tr_px = gerar_imagens(2000, seed=0)     # gatos e cachorros CENTRALIZADOS
X_te_px, y_te_px = gerar_imagens(1000, seed=1)
print("X tem forma", X_tr_px.shape, "-> uma coluna por pixel")

logit_px = LogisticRegression(max_iter=2000, C=0.1).fit(X_tr_px, y_tr_px)
print(f"Acurácia no teste (imagens centralizadas): {logit_px.score(X_te_px, y_te_px):.1%}")

Acertou praticamente tudo. Ótimo? Vamos olhar **o que** o modelo aprendeu: cada pixel tem um $\beta_j$, e podemos desenhar os 576 coeficientes como uma imagem.

In [ ]:
pesos = logit_px.coef_.reshape(N, N)          # os 576 β de volta ao formato de imagem
lim = np.abs(pesos).max()

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(pesos, cmap="RdBu_r", vmin=-lim, vmax=lim)   # mapa divergente: zero = branco
fig.colorbar(im, ax=ax, label="β do pixel")
ax.set_title("Os 576 coeficientes do logit")
ax.axis("off")
plt.show()

Vermelho: se o pixel for claro, a chance de **gato** sobe; azul: sobe a de **cachorro**. Repare nas laterais vermelhas: ali ficam as **orelhas escuras do cachorro**. O modelo aprendeu "claro onde o cachorro teria orelha → gato", ou seja, reconhece o gato pela **ausência** do cachorro, e sempre na mesma posição. $\beta_{295}$ não é "o efeito do brilho de um pixel": é um pedaço de um molde preso ao lugar.

### E se o gato andar alguns pixels para o lado?

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, d in zip(axs, [0, 2, 4, 6]):
    ax.imshow(desenhar("gato", dx=d), cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"deslocado {d} px")
    ax.axis("off")
plt.show()

In [ ]:
deslocs = range(0, 9)
# *gerar_imagens(...) desempacota o par (X, y) nos dois argumentos de .score
acc_centro = [logit_px.score(*gerar_imagens(600, desloc=d, seed=100 + d)) for d in deslocs]

# E se o treino tiver gatos e cachorros deslocados até 4 px, em qualquer direção?
logit_desl = LogisticRegression(max_iter=2000, C=0.1).fit(
    *gerar_imagens(4000, desloc=4, aleatorio=True, seed=7))
acc_desl = [logit_desl.score(*gerar_imagens(600, desloc=d, seed=200 + d)) for d in deslocs]

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.axvspan(-0.3, 4.3, color=CORES["laranja"], alpha=0.08)
ax.text(2, 0.06, "deslocamentos vistos no 2º treino", ha="center", color=CORES["laranja"])
ax.plot(deslocs, acc_centro, "o-", label="treino só com imagens centralizadas")
ax.plot(deslocs, acc_desl, "s-", label="treino com deslocamentos de até 4 px")
ax.axhline(0.5, color=CORES["cinza"], ls=":", lw=1.5)
ax.text(8, 0.515, "chute (50%)", ha="right", color=CORES["cinza"])
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.set(title="O logit decora posições", xlabel="deslocamento no teste (pixels)",
       ylabel="acurácia", ylim=(-0.02, 1.04))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)   # legenda abaixo do gráfico
plt.show()

for d, a, b in zip(deslocs, acc_centro, acc_desl):
    print(f"{d} px: {a:.0%} | {b:.0%}")

### O que aprendemos
- O logit acerta tudo quando o gato está **onde ele estava no treino** e despenca quando ele anda alguns pixels.
- Com 6 px ou mais, o segundo modelo erra **quase todas**: pior que chutar. O gato deslocado cobre a região onde o cachorro tem orelha escura, e o modelo responde "cachorro" com confiança.
- Mostrar gatos deslocados no treino ajuda **só dentro da faixa vista**: o modelo continua decorando posições, agora mais posições. Para cobrir posição, tamanho, luz, pose e raça, precisaríamos de exemplos de **todas** as combinações.
- É para isso que existem modelos que aprendem padrões **locais** e os reconhecem em qualquer lugar da imagem (as redes convolucionais). A ideia de rede neural é tema da Aula 5.

> Honestidade: nossas imagens são de brinquedo. Com fotos reais, o problema é muito mais difícil, e o logit, muito pior.